In [1]:
!pip install -q -U sentence-transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.5 MB/s eta 0:00:00


In [2]:
import torch
import random
import numpy as np
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator

In [3]:
# ---------------------------------------------------------
# 1. Setup & Reproducibility
# ---------------------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

#MODEL_ID = "BAAI/bge-small-en-v1.5"
#MAX_SEQ_LENGTH = 384
MODEL_ID = "sentence-transformers/all-MiniLM-L12-v2"
MAX_SEQ_LENGTH = 256 # MiniLM is natively optimized for 256 tokens
BATCH_SIZE = 32 # Fits easily on Colab T4 for a small model
EPOCHS = 2

In [4]:
# ---------------------------------------------------------
# 2. Dataset Preparation
# ---------------------------------------------------------
print("Loading datasets...")
# Load unlabeled data for training and labeled data for evaluation
train_dataset_raw = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled", split="train")
eval_dataset_raw = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

# Shuffle and sample 10,000 records
train_dataset_raw = train_dataset_raw.shuffle(seed=42).select(range(10000))

# Preprocess training data into pairs: {"anchor": question, "positive": context}
def format_train_data(example):
    return {
        "anchor": example["question"],
        "positive": " ".join(example["context"]["contexts"])
    }

train_dataset = train_dataset_raw.map(
    format_train_data,
    remove_columns=train_dataset_raw.column_names # Remove raw columns
)

Loading datasets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pqa_unlabeled/train-00000-of-00001.parqu(…):   0%|          | 0.00/66.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61249 [00:00<?, ? examples/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [5]:
# ---------------------------------------------------------
# 3. Evaluation Pipeline Setup
# ---------------------------------------------------------
print("Building Evaluation Corpus...")
corpus = {}          # doc_id -> document_text
queries = {}         # query_id -> query_text
relevant_docs = {}   # query_id -> set([doc_id])

context_to_id = {}
doc_idx = 0

# Extract queries and construct unique corpus from the eval split
for i, row in enumerate(eval_dataset_raw):
    q_id = f"q_{i}"
    queries[q_id] = row["question"]

    # Concatenate context
    context_str = " ".join(row["context"]["contexts"])

    # Avoid duplicate contexts in corpus
    if context_str not in context_to_id:
        doc_id = f"d_{doc_idx}"
        corpus[doc_id] = context_str
        context_to_id[context_str] = doc_id
        doc_idx += 1
    else:
        doc_id = context_to_id[context_str]

    relevant_docs[q_id] = set([doc_id])

evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="pubmedqa-eval",
    mrr_at_k=[10],
    precision_recall_at_k=[5, 10], # Extracts Recall@5 and Recall@10
    show_progress_bar=True
)

Building Evaluation Corpus...


In [6]:
# ---------------------------------------------------------
# 4. Model Loading & Base Evaluation
# ---------------------------------------------------------
print(f"Loading base model: {MODEL_ID}")
model = SentenceTransformer(MODEL_ID)
model.max_seq_length = MAX_SEQ_LENGTH

print("\n--- Evaluating Base (Pre-trained) Model ---")
base_metrics = evaluator(model, output_path="./eval_base")
print(f"Base Model Results: {base_metrics}")

Loading base model: sentence-transformers/all-MiniLM-L12-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:03<00:00,  3.87s/it]

Base Model Results: {'pubmedqa-eval_cosine_accuracy@1': 0.976, 'pubmedqa-eval_cosine_accuracy@3': 0.987, 'pubmedqa-eval_cosine_accuracy@5': 0.991, 'pubmedqa-eval_cosine_accuracy@10': 0.993, 'pubmedqa-eval_cosine_precision@5': 0.19820000000000004, 'pubmedqa-eval_cosine_precision@10': 0.09930000000000001, 'pubmedqa-eval_cosine_recall@5': 0.991, 'pubmedqa-eval_cosine_recall@10': 0.993, 'pubmedqa-eval_cosine_ndcg@10': 0.9850258408869751, 'pubmedqa-eval_cosine_mrr@10': 0.9824, 'pubmedqa-eval_cosine_map@100': 0.9827266950440864}


In [7]:
from huggingface_hub import login
login() # This will prompt you to paste your token

In [8]:
# ---------------------------------------------------------
# 5. Training Objective & Setup
# ---------------------------------------------------------
# InfoNCE Loss: Uses anchor + positive pairs, and treats all other positives in the batch as negatives
train_loss = losses.MultipleNegativesRankingLoss(model)

# Define Hugging Face Training Arguments
args = SentenceTransformerTrainingArguments(
    output_dir="./minilm-pubmedqa-finetuned",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    fp16=True, # Enable mixed precision for speed and memory savings
    eval_strategy="epoch",    # Evaluate at the end of each epoch
    save_strategy="epoch",
    logging_steps=50,
    seed=42,
    push_to_hub=True, # <-- Add this line
    hub_model_id="AswiniSivakumar/mini-pubmedqa-finetuned" # <-- Add your HF username
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [9]:
# ---------------------------------------------------------
# 6. Fine-Tuning
# ---------------------------------------------------------
print("\n--- Starting Contrastive Fine-Tuning ---")
trainer.train()


--- Starting Contrastive Fine-Tuning ---


Epoch,Training Loss,Validation Loss,Pubmedqa-eval Cosine Accuracy@1,Pubmedqa-eval Cosine Accuracy@3,Pubmedqa-eval Cosine Accuracy@5,Pubmedqa-eval Cosine Accuracy@10,Pubmedqa-eval Cosine Precision@5,Pubmedqa-eval Cosine Precision@10,Pubmedqa-eval Cosine Recall@5,Pubmedqa-eval Cosine Recall@10,Pubmedqa-eval Cosine Ndcg@10,Pubmedqa-eval Cosine Mrr@10,Pubmedqa-eval Cosine Map@100
1,0.039833,No log,0.980000,0.990000,0.992000,0.997000,0.198400,0.099700,0.992000,0.997000,0.988672,0.986017,0.986081
2,0.011953,No log,0.981000,0.990000,0.992000,0.995000,0.198400,0.099500,0.992000,0.995000,0.988498,0.986376,0.986608


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

...

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:01<00:00,  1.57s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=626, training_loss=0.02033533265415472, metrics={'train_runtime': 157.9146, 'train_samples_per_second': 126.651, 'train_steps_per_second': 3.964, 'total_flos': 0.0, 'train_loss': 0.02033533265415472, 'epoch': 2.0})

In [10]:
# ---------------------------------------------------------
# 7. Save & Final Evaluation
# ---------------------------------------------------------
print("\n--- Training Complete! Saving Model ---")
model.save("./minilm-pubmedqa-finetuned/final")

print("\n--- Evaluating Fine-Tuned Model ---")
finetuned_metrics = evaluator(model, output_path="./eval_finetuned")
print(f"Fine-Tuned Model Results: {finetuned_metrics}")


--- Training Complete! Saving Model ---


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Evaluating Fine-Tuned Model ---


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:01<00:00,  1.52s/it]

Fine-Tuned Model Results: {'pubmedqa-eval_cosine_accuracy@1': 0.981, 'pubmedqa-eval_cosine_accuracy@3': 0.99, 'pubmedqa-eval_cosine_accuracy@5': 0.992, 'pubmedqa-eval_cosine_accuracy@10': 0.995, 'pubmedqa-eval_cosine_precision@5': 0.19840000000000002, 'pubmedqa-eval_cosine_precision@10': 0.09950000000000002, 'pubmedqa-eval_cosine_recall@5': 0.992, 'pubmedqa-eval_cosine_recall@10': 0.995, 'pubmedqa-eval_cosine_ndcg@10': 0.9884978211041615, 'pubmedqa-eval_cosine_mrr@10': 0.9863761904761904, 'pubmedqa-eval_cosine_map@100': 0.986607536157536}


In [11]:
from sentence_transformers import SentenceTransformer

# Load your custom, fine-tuned medical model
model = SentenceTransformer("./minilm-pubmedqa-finetuned/final")

# Generate an embedding for a user query
query_embedding = model.encode("What are the side effects of Lisinopril?")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [12]:
print(query_embedding)

[-1.06839612e-01 -5.37125766e-02 -3.15358043e-02  1.07663758e-01
 -1.46307249e-03 -6.22187108e-02  1.17969833e-01  3.48824337e-02
 -4.73892912e-02 -7.58608878e-02 -8.12459178e-03  6.52870759e-02
 -1.53094213e-02 -2.81664617e-02  2.40520388e-03 -2.30326280e-02
  2.26843357e-02  4.36126515e-02 -2.04611756e-02 -1.75791290e-02
  4.58297208e-02 -3.80830392e-02  7.21675530e-02 -7.07121007e-03
 -7.75490552e-02  2.87495591e-02 -1.33520772e-03 -2.07688976e-02
  4.79643792e-02  6.28320314e-03  1.56449247e-02  1.44639805e-01
 -2.68367864e-02 -9.92382020e-02 -2.32163705e-02  3.71022150e-02
 -3.31529379e-02 -7.07233138e-03  1.40120853e-02 -1.59507226e-02
  6.08128272e-02 -3.15454579e-03  4.01631230e-03  7.56520964e-03
 -2.13208310e-02 -6.23482540e-02  2.00377684e-02 -2.44937092e-02
  7.42654577e-02  1.30796907e-02 -2.55308091e-03 -8.22334588e-02
  9.96899828e-02 -2.54159216e-02 -1.85838211e-02 -7.68044591e-03
 -5.72973713e-02 -2.41701696e-02 -4.54608425e-02  9.91252884e-02
 -3.09468545e-02  7.61860